In [1]:
from pyspark.sql import SparkSession,functions as F
from pyspark.sql.types import (
    StructType,StructField,
    IntegerType,StringType,
    TimestampType,DecimalType
)
from pathlib import Path
from pyspark.sql.window import Window

spark = SparkSession.builder.\
appName("FestiveDataDetective").\
master("local[*]").\
getOrCreate()

In [2]:
#loading csv files with no schemainference to understand what the files actually contain 
#this cell is performed to define the explicit schemas for all datasets by observing column names and data it holds
DATA_DIR = Path('Data/raw')

files = {
    'attendees':"attendees.csv",
    'food_orders':"food_orders.csv",
    "food_stalls":"food_stalls.csv",
    "incidents":"incidents.csv",
    "performances":"performances.csv",
    "stages":"stages.csv",
    "ticket_scans":"ticket_scans.csv"
}
raw = {}

for name,filename in files.items():
    filepath = DATA_DIR / filename

    raw[name] = (
        spark.read.
        option('header',True)
        .option('inferSchema',False)
        .option('mode','PERMISSIVE')
        .csv(str(DATA_DIR / filename))
    )
print("Successfully read",len(raw),"csv files")

Successfully read 7 csv files


In [3]:
#displaying row count,column details,5 records and understading the schema

for name, df in raw.items():
    print()
    print("=" * 50)
    print()
    print(name.upper())
    print("Number of rows: ",df.count())
    print(f"columns({len(df.columns)}): {df.columns}")
    print("-"*50)
    df.printSchema()
    df.show(5,truncate=False)



ATTENDEES
Number of rows:  1000
columns(4): ['attendee_id', 'ticket_type', 'age_group', 'home_city']
--------------------------------------------------
root
 |-- attendee_id: string (nullable = true)
 |-- ticket_type: string (nullable = true)
 |-- age_group: string (nullable = true)
 |-- home_city: string (nullable = true)

+-----------+-----------+---------+---------+
|attendee_id|ticket_type|age_group|home_city|
+-----------+-----------+---------+---------+
|A00001     |VIP        |25-34    |Bengaluru|
|A00002     |Student    |18-24    |Mumbai   |
|A00003     |general    |25-34    |Hyderabad|
|A00004     |General    |NULL     |Hyderabad|
|A00005     |VIP        |18-24    |Hyderabad|
+-----------+-----------+---------+---------+
only showing top 5 rows


FOOD_ORDERS
Number of rows:  1602
columns(6): ['order_id', 'attendee_id', 'stall_id', 'order_time', 'amount', 'order_status']
--------------------------------------------------
root
 |-- order_id: string (nullable = true)
 |-- atten

In [4]:
#defining explicit schemas for given datasets
schemas ={
    "attendees":StructType([
        StructField("attendee_id",StringType(),True),
        StructField("ticket_type",StringType(),True),
        StructField("age_group",StringType(),True),
        StructField("home_city",StringType(),True)
    ]),
    "food_orders":StructType([
        StructField("order_id",StringType(),True),
        StructField("attendee_id",StringType(),True),
        StructField("stall_id",StringType(),True),
        StructField("order_time",TimestampType(),True),
        StructField("amount",DecimalType(10,2),True),
        StructField("order_status",StringType(),True)
    ]),
    "food_stalls":StructType([
        StructField("stall_id",StringType(),True),
        StructField('stall_name',StringType(),True),
        StructField('cuisine',StringType(),True),
        StructField("stage_id",StringType(),True)
    ]),
    'incidents':StructType([
        StructField("incident_id",StringType(),True),
        StructField("stage_id",StringType(),True),
        StructField("incident_time",TimestampType(),True),
        StructField("incident_type",StringType(),True),
        StructField("severity",StringType(),True)
    ]),
    "performances":StructType([
        StructField("performance_id",StringType(),True),
        StructField("artist_name",StringType(),True),
        StructField('stage_id',StringType(),True),
        StructField('start_time',TimestampType(),True),
        StructField('end_time',TimestampType(),True),
        StructField('genre',StringType(),True)
    ]),
    "stages":StructType([
        StructField("stage_id",StringType(),True),
        StructField("stage_name",StringType(),True),
        StructField("capacity",IntegerType(),True),
        StructField('zone',StringType(),True)
    ]),
    "ticket_scans":StructType([
        StructField("scan_id",StringType(),True),
        StructField("attendee_id",StringType(),True),
        StructField("stage_id",StringType(),True),
        StructField("scan_time",TimestampType(),True),
        StructField("scan_status",StringType(),True)
    ])
}

assert set(schemas) == set(files)
print("Schemas defined: ",len(schemas))

Schemas defined:  7


In [5]:
#reading datasets using defined schemas

typed = {}

for name,filename in files.items():
    typed[name] = (
        spark.read
        .schema(schemas[name])
        .option("header",True)
        .option("mode",'PERMISSIVE')
        .option('timestampFormat','yyyy-MM-dd HH:mm:ss')
        .csv(str(DATA_DIR/filename))
        
    )

print("Typed Datasets loaded: ",len(typed))

# Optional aliases for dataset-specific analysis

attendees_df = typed["attendees"]
ticket_scans_df = typed["ticket_scans"]
food_orders_df = typed["food_orders"]
stages_df = typed["stages"]
performances_df = typed["performances"]
food_stalls_df = typed["food_stalls"]
incidents_df = typed["incidents"]

Typed Datasets loaded:  7


In [6]:
#finding total row counts, missing values of each column for each dataset
missing_value_reports = {}

for name,df in typed.items():
    print(f'\n{'=' * 60}')
    print(name.upper())

    expressions = [F.count("*").alias("row_count")]

    for column in df.columns:
        expressions.append(
            F.sum(
                    F.when(
                        F.col(column).isNull() |
                        (F.trim(F.col(column).cast('string'))==""), 1
                    ).otherwise(0)
                ).alias(f'{column}_missing')
        )
    report = df.agg(*expressions)
    missing_value_reports[name] = report
    report.show(truncate = False)



ATTENDEES
+---------+-------------------+-------------------+-----------------+-----------------+
|row_count|attendee_id_missing|ticket_type_missing|age_group_missing|home_city_missing|
+---------+-------------------+-------------------+-----------------+-----------------+
|1000     |0                  |0                  |154              |266              |
+---------+-------------------+-------------------+-----------------+-----------------+


FOOD_ORDERS
+---------+----------------+-------------------+----------------+------------------+--------------+--------------------+
|row_count|order_id_missing|attendee_id_missing|stall_id_missing|order_time_missing|amount_missing|order_status_missing|
+---------+----------------+-------------------+----------------+------------------+--------------+--------------------+
|1602     |0               |0                  |0               |0                 |39            |0                   |
+---------+----------------+-------------------+---

In [7]:
#identifying the duplicate key groups and storing them in dictonaries

primary_keys = {
    "attendees":'attendee_id',
    "food_orders":"order_id",
    "food_stalls":"stall_id",
    "incidents":"incident_id",
    "performances":"performance_id",
    "stages":"stage_id",
    "ticket_scans":"scan_id"
}
duplicate_key_reports = {}
for name, key in primary_keys.items():
    duplicate_groups = (
        typed[name].groupBy(key)
        .count()
        .filter(
            F.col(key).isNotNull() & 
            (F.col("count") > 1)
        )
    )
    duplicate_key_reports[name] = duplicate_groups
    print(f'\n{'='*50}')
    print(f'Dataset: {name.upper()}')
    print(f'Primary key: {key}')
    duplicate_groups.show(30,truncate=False)
    print(
        "Number of duplicate key groups:",
        duplicate_groups.count()
    )


Dataset: ATTENDEES
Primary key: attendee_id
+-----------+-----+
|attendee_id|count|
+-----------+-----+
+-----------+-----+

Number of duplicate key groups: 0

Dataset: FOOD_ORDERS
Primary key: order_id
+--------+-----+
|order_id|count|
+--------+-----+
|O00006  |2    |
|O00056  |2    |
+--------+-----+

Number of duplicate key groups: 2

Dataset: FOOD_STALLS
Primary key: stall_id
+--------+-----+
|stall_id|count|
+--------+-----+
+--------+-----+

Number of duplicate key groups: 0

Dataset: INCIDENTS
Primary key: incident_id
+-----------+-----+
|incident_id|count|
+-----------+-----+
+-----------+-----+

Number of duplicate key groups: 0

Dataset: PERFORMANCES
Primary key: performance_id
+--------------+-----+
|performance_id|count|
+--------------+-----+
+--------------+-----+

Number of duplicate key groups: 0

Dataset: STAGES
Primary key: stage_id
+--------+-----+
|stage_id|count|
+--------+-----+
+--------+-----+

Number of duplicate key groups: 0

Dataset: TICKET_SCANS
Primary k

In [8]:
#displaying the value counts(frequency of distinct values) of each dataset
category_columns = {
    "attendees": ["ticket_type", "age_group", "home_city"],
    "food_orders": ["order_status"],
    "food_stalls": ["cuisine"],
    "incidents": ["incident_type", "severity"],
    "performances": ["genre"],
    "stages": ["zone"],
    "ticket_scans": ["scan_status"],
}

for name, columns in category_columns.items():
    print(f"\n{'=' * 55}")
    print(name.upper())

    for column in columns:
        print(f"\nValues in {column}:")

        (
            typed[name]
            .groupBy(F.lower(F.trim(F.col(column).cast("string"))).alias("normalized_value"))
            .count()
            .orderBy(F.desc("count"))
            .show(30, truncate=False)
        )





ATTENDEES

Values in ticket_type:
+----------------+-----+
|normalized_value|count|
+----------------+-----+
|general         |322  |
|vip             |321  |
|student         |186  |
|day pass        |171  |
+----------------+-----+


Values in age_group:
+----------------+-----+
|normalized_value|count|
+----------------+-----+
|unknown         |182  |
|45+             |182  |
|25-34           |161  |
|18-24           |161  |
|35-44           |160  |
|NULL            |154  |
+----------------+-----+


Values in home_city:
+----------------+-----+
|normalized_value|count|
+----------------+-----+
|NULL            |266  |
|delhi           |130  |
|mumbai          |123  |
|pune            |122  |
|chennai         |121  |
|bengaluru       |121  |
|hyderabad       |117  |
+----------------+-----+


FOOD_ORDERS

Values in order_status:
+----------------+-----+
|normalized_value|count|
+----------------+-----+
|completed       |607  |
|pending         |374  |
|cancelled       |317  |
|refu

In [9]:
#displaying statstical values for food_orders and Stages .Finding invalid times in Performances
print("Food orders - Amount distribution")
food_orders_df.select(
    F.min("amount").alias("min_amount"),
    F.max("amount").alias("max_amount"),
    F.sum(
        F.when(F.col("amount")<0,1)
        .otherwise(0)
    ).alias("negative_amounts"),
    F.sum(
        F.when(F.col("amount")==0,1)
        .otherwise(0)
    ).alias("zero_amounts")
).show()

print("Stages capacity distribution")
stages_df.select(
    F.min("capacity").alias("min_capacity"),
    F.max("capacity").alias("max_capacity"),
    F.sum(
        F.when(F.col("capacity")<=0,1)
        .otherwise(0)
    ).alias("non-positive_capacity")
).show()

print("Performances - Invalid time ranges")
performances_df.filter(
    F.col("start_time").isNotNull()
    & F.col("end_time").isNotNull()
    & (F.col("end_time") <= F.col("start_time"))
).show(truncate=False)

Food orders - Amount distribution
+----------+----------+----------------+------------+
|min_amount|max_amount|negative_amounts|zero_amounts|
+----------+----------+----------------+------------+
|    -12.50|     54.97|               1|           0|
+----------+----------+----------------+------------+

Stages capacity distribution
+------------+------------+---------------------+
|min_capacity|max_capacity|non-positive_capacity|
+------------+------------+---------------------+
|        1200|        8500|                    0|
+------------+------------+---------------------+

Performances - Invalid time ranges
+--------------+-----------+--------+----------+--------+-----+
|performance_id|artist_name|stage_id|start_time|end_time|genre|
+--------------+-----------+--------+----------+--------+-----+
+--------------+-----------+--------+----------+--------+-----+



In [10]:
#displaying duplicate records for datasets which have found earlier(based on duplicate groups)
# 1. Inspect repeated food order IDs
print("DUPLICATE FOOD ORDERS")

duplicate_order_ids = (
    typed["food_orders"]
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .select("order_id")
)

(
    typed["food_orders"]
    .join(duplicate_order_ids, on="order_id", how="inner")
    .orderBy("order_id")
    .show(truncate=False)
)

# 2. Inspect repeated scan IDs
print("DUPLICATE TICKET SCANS")

duplicate_scan_ids = (
    typed["ticket_scans"]
    .groupBy("scan_id")
    .count()
    .filter(F.col("count") > 1)
    .select("scan_id")
)

(
    typed["ticket_scans"]
    .join(duplicate_scan_ids, on="scan_id", how="inner")
    .orderBy("scan_id")
    .show(truncate=False)
)

# 3. Inspect negative food amounts
print("NEGATIVE FOOD AMOUNTS")

(
    typed["food_orders"]
    .filter(F.col("amount") < 0)
    .show(truncate=False)
)


DUPLICATE FOOD ORDERS
+--------+-----------+--------+-------------------+------+------------+
|order_id|attendee_id|stall_id|order_time         |amount|order_status|
+--------+-----------+--------+-------------------+------+------------+
|O00006  |A00608     |FS02    |2026-09-13 20:35:00|43.07 |pending     |
|O00006  |A00608     |FS02    |2026-09-13 20:35:00|43.07 |pending     |
|O00056  |A00091     |FS05    |2026-09-11 18:56:00|20.87 |pending     |
|O00056  |A00091     |FS05    |2026-09-11 18:56:00|20.87 |pending     |
+--------+-----------+--------+-------------------+------+------------+

DUPLICATE TICKET SCANS
+--------+-----------+--------+-------------------+-----------+
|scan_id |attendee_id|stage_id|scan_time          |scan_status|
+--------+-----------+--------+-------------------+-----------+
|SC000016|A00002     |ST02    |2026-09-12 13:08:00|valid      |
|SC000016|A00002     |ST02    |2026-09-12 13:08:00|valid      |
|SC000029|A00522     |ST02    |2026-09-13 18:18:00|valid  

In [11]:
#checking referntial integrity - a record from one dataset refering  an entity from another dataset.
#this code block indentifies records which refers entities which doesnt exsists: orphan records
def show_orphan_records(source_df,source_key,target_df,target_key,relationship_name):
    target_keys = (
        target_df.select(F.col(target_key).alias("_target_key"))
        .filter(F.col("_target_key").isNotNull())
        .distinct()
    )

    orphans = (
        source_df.filter(F.col(source_key).isNotNull())
        .join(target_keys,
              F.col(source_key)==F.col('_target_key'),
              "left_anti")
    )

    print(f'\n{relationship_name}')
    print(f'orphan records count: {orphans.count()}')
    orphans.show(30,truncate=False)


show_orphan_records(
    typed["food_orders"], "attendee_id",
    typed["attendees"], "attendee_id",
    "Food orders -> attendees"
)

show_orphan_records(
    typed["food_orders"], "stall_id",
    typed["food_stalls"], "stall_id",
    "Food orders -> food stalls"
)

show_orphan_records(
    typed["ticket_scans"], "attendee_id",
    typed["attendees"], "attendee_id",
    "Ticket scans -> attendees"
)

show_orphan_records(
    typed["ticket_scans"], "stage_id",
    typed["stages"], "stage_id",
    "Ticket scans -> stages"
)

show_orphan_records(
    typed["performances"], "stage_id",
    typed["stages"], "stage_id",
    "Performances -> stages"
)

show_orphan_records(
    typed["incidents"], "stage_id",
    typed["stages"], "stage_id",
    "Incidents -> stages"
)


Food orders -> attendees
orphan records count: 0
+--------+-----------+--------+----------+------+------------+
|order_id|attendee_id|stall_id|order_time|amount|order_status|
+--------+-----------+--------+----------+------+------------+
+--------+-----------+--------+----------+------+------------+


Food orders -> food stalls
orphan records count: 1
+--------+-----------+--------+-------------------+------+------------+
|order_id|attendee_id|stall_id|order_time         |amount|order_status|
+--------+-----------+--------+-------------------+------+------------+
|O00026  |A00118     |FS99    |2026-09-13 21:31:00|4.07  |cancelled   |
+--------+-----------+--------+-------------------+------+------------+


Ticket scans -> attendees
orphan records count: 0
+-------+-----------+--------+---------+-----------+
|scan_id|attendee_id|stage_id|scan_time|scan_status|
+-------+-----------+--------+---------+-----------+
+-------+-----------+--------+---------+-----------+


Ticket scans -> sta

In [12]:

# 1. Normalize whitespace and convert blank strings to NULL.
standardized = {}

for name, df in typed.items():
    result = df

    for column_name, data_type in result.dtypes:
        if data_type == "string":
            result = result.withColumn(
                column_name,
                F.when(
                    F.trim(F.col(column_name)) == "",
                    F.lit(None).cast("string")
                ).otherwise(F.trim(F.col(column_name)))
            )

    standardized[name] = result


# 2. Normalize the categorical columns identified during profiling.
category_columns = {
    "attendees": ["ticket_type", "age_group", "home_city"],
    "food_orders": ["order_status"],
    "food_stalls": ["cuisine"],
    "incidents": ["incident_type", "severity"],
    "performances": ["genre"],
    "stages": ["zone"],
    "ticket_scans": ["scan_status"],
}

for name, columns in category_columns.items():
    result = standardized[name]

    for column_name in columns:
        result = result.withColumn(
            column_name,
            F.lower(F.col(column_name))
        )

    standardized[name] = result



print("Standardization completed.")

standardized["food_orders"].groupBy("order_status").count().show(
    truncate=False
)


Standardization completed.
+------------+-----+
|order_status|count|
+------------+-----+
|refunded    |304  |
|completed   |607  |
|cancelled   |317  |
|pending     |374  |
+------------+-----+



In [13]:
#seperating clean and quarantine data
source_data = standardized
# Expected primary keys for each dataset
primary_keys = {
    "attendees": "attendee_id",
    "food_orders": "order_id",
    "food_stalls": "stall_id",
    "incidents": "incident_id",
    "performances": "performance_id",
    "stages": "stage_id",
    "ticket_scans": "scan_id",
}


# 1. Identify duplicate primary keys
duplicate_keys = {}

for name, key in primary_keys.items():
    duplicate_keys[name] = (
        source_data[name]
        .groupBy(key)
        .count()
        .filter(
            F.col(key).isNotNull()
            & (F.trim(F.col(key)) != "")
            & (F.col("count") > 1)
        )
        .select(key)
    )


# 2. Build reference-key DataFrames
# Exclude null and blank IDs from the reference datasets.

attendee_keys = (
    source_data["attendees"]
    .select(F.col("attendee_id").alias("_ref_id"))
    .filter(
        F.col("_ref_id").isNotNull()
        & (F.trim(F.col("_ref_id")) != "")
    )
    .distinct()
)

stall_keys = (
    source_data["food_stalls"]
    .select(F.col("stall_id").alias("_ref_id"))
    .filter(
        F.col("_ref_id").isNotNull()
        & (F.trim(F.col("_ref_id")) != "")
    )
    .distinct()
)

stage_keys = (
    source_data["stages"]
    .select(F.col("stage_id").alias("_ref_id"))
    .filter(
        F.col("_ref_id").isNotNull()
        & (F.trim(F.col("_ref_id")) != "")
    )
    .distinct()
)


# 3. Find non-null, non-blank references that do not exist in the lookup

def unmatched_nonnullKeys(df, column, reference_keys):
    """Return rows with a non-empty reference key that has no match."""

    return (
        df.filter(
            F.col(column).isNotNull()
            & (F.trim(F.col(column)) != "")
        )
        .join(
            reference_keys,
            F.col(column) == F.col("_ref_id"),
            "left_anti"
        )
    )


# 4. Define required-field and business-rule validation

invalid_rules = {
    "attendees": lambda df: (
        F.col("attendee_id").isNull()
        | (F.trim(F.col("attendee_id")) == "")
    ),

    "food_orders": lambda df: (
        F.col("order_id").isNull()
        | (F.trim(F.col("order_id")) == "")
        | F.col("attendee_id").isNull()
        | (F.trim(F.col("attendee_id")) == "")
        | F.col("stall_id").isNull()
        | (F.trim(F.col("stall_id")) == "")
        | F.col("order_time").isNull()
        | F.col("amount").isNull()
        | (F.col("amount") < 0)
        | F.col("order_status").isNull()
    ),

   "food_stalls": lambda df: (
    F.col("stall_id").isNull()
    | (F.trim(F.col("stall_id")) == "")
    | F.col("stage_id").isNull()
    | (F.trim(F.col("stage_id")) == "")
    ),

    "incidents": lambda df: (
        F.col("incident_id").isNull()
        | (F.trim(F.col("incident_id")) == "")
        | F.col("stage_id").isNull()
        | (F.trim(F.col("stage_id")) == "")
        | F.col("incident_time").isNull()
        | F.col("incident_type").isNull()
        | (F.trim(F.col("incident_type")) == "")
    ),

    "performances": lambda df: (
        F.col("performance_id").isNull()
        | (F.trim(F.col("performance_id")) == "")
        | F.col("stage_id").isNull()
        | (F.trim(F.col("stage_id")) == "")
        | F.col("start_time").isNull()
        | F.col("end_time").isNull()
        | (F.col("end_time") <= F.col("start_time"))
        | F.col("artist_name").isNull()
        | (F.trim(F.col("artist_name")) == "")
    ),

    "stages": lambda df: (
        F.col("stage_id").isNull()
        | (F.trim(F.col("stage_id")) == "")
        | F.col("capacity").isNull()
        | (F.col("capacity") <= 0)
    ),

    "ticket_scans": lambda df: (
        F.col("scan_id").isNull()
        | (F.trim(F.col("scan_id")) == "")
        | F.col("attendee_id").isNull()
        | (F.trim(F.col("attendee_id")) == "")
        | F.col("stage_id").isNull()
        | (F.trim(F.col("stage_id")) == "")
        | F.col("scan_time").isNull()
    ),
}


# 5. Define foreign-key/reference-integrity rules

reference_rules = {
    "food_orders": [
        ("attendee_id", attendee_keys),
        ("stall_id", stall_keys),
    ],

    "ticket_scans": [
        ("attendee_id", attendee_keys),
        ("stage_id", stage_keys),
    ],

    "performances": [
        ("stage_id", stage_keys),
    ],

    "food_stalls": [
    ("stage_id", stage_keys),
    ],

    "incidents": [
        ("stage_id", stage_keys),
    ],
}


# 6. Create clean and quarantine DataFrames

clean = {}
quarantine = {}

for name, df in source_data.items():

    # Evaluate base validation rules
    base_invalid = invalid_rules[name](df)

    key = primary_keys[name]

    # All rows belonging to duplicate primary-key groups
    duplicate_rows = df.join(
        duplicate_keys[name],
        on=key,
        how="left_semi"
    )

    # Rows whose primary keys are not duplicated
    non_duplicate_rows = df.join(
        duplicate_keys[name],
        on=key,
        how="left_anti"
    )

    # Invalid rows that fail required-field/business rules
    base_bad = non_duplicate_rows.filter(base_invalid)

    # Continue validation only with rows that pass base rules
    remaining = non_duplicate_rows.filter(~base_invalid)

    # Start collecting quarantined rows and their reasons
    invalid_parts = [
        duplicate_rows.withColumn(
            "quarantine_reason",
            F.lit("duplicate_primary_key")
        ),

        base_bad.withColumn(
            "quarantine_reason",
            F.lit("required_field_or_value_invalid")
        ),
    ]

    # Check each foreign-key relationship
    for fk_column, reference_df in reference_rules.get(name, []):

        # Find rows whose reference key does not exist
        bad_fk = unmatched_nonnullKeys(
            remaining,
            fk_column,
            reference_df
        )

        # Quarantine unmatched references
        invalid_parts.append(
            bad_fk.withColumn(
                "quarantine_reason",
                F.lit(f"unmatched_reference:{fk_column}")
            )
        )

        # Keep only rows with a matching reference for the next check
        remaining = remaining.join(
            reference_df,
            F.col(fk_column) == F.col("_ref_id"),
            "left_semi"
        )

    # Combine all quarantine categories into one DataFrame
    quarantine[name] = invalid_parts[0]

    for part in invalid_parts[1:]:
        quarantine[name] = quarantine[name].unionByName(
            part,
            allowMissingColumns=True
        )

    # Rows that passed all applicable validations
    clean[name] = remaining


print("Clean and quarantine DataFrames created.")

Clean and quarantine DataFrames created.


In [14]:

# Clean and quarantine counts for each dataset

for name in standardized:
    key = primary_keys[name]

    print(f"\n{'=' * 40}")
    print(name.upper())
    print("Raw rows:", raw[name].count())
    print("Clean rows:", clean[name].count())
    print("Quarantined rows:", quarantine[name].count())

    print("\nQuarantine reasons:")
    quarantine[name].groupBy("quarantine_reason").count().show(
        truncate=False
    )

    missing_key_count = (
        clean[name]
        .filter(
            F.col(key).isNull()
            | (F.trim(F.col(key)) == "")
        )
        .count()
    )

    print(f"Missing primary keys: {missing_key_count}")



ATTENDEES
Raw rows: 1000
Clean rows: 1000
Quarantined rows: 0

Quarantine reasons:
+-----------------+-----+
|quarantine_reason|count|
+-----------------+-----+
+-----------------+-----+

Missing primary keys: 0

FOOD_ORDERS
Raw rows: 1602
Clean rows: 1557
Quarantined rows: 45

Quarantine reasons:
+-------------------------------+-----+
|quarantine_reason              |count|
+-------------------------------+-----+
|duplicate_primary_key          |4    |
|required_field_or_value_invalid|40   |
|unmatched_reference:stall_id   |1    |
+-------------------------------+-----+

Missing primary keys: 0

FOOD_STALLS
Raw rows: 6
Clean rows: 6
Quarantined rows: 0

Quarantine reasons:
+-----------------+-----+
|quarantine_reason|count|
+-----------------+-----+
+-----------------+-----+

Missing primary keys: 0

INCIDENTS
Raw rows: 180
Clean rows: 178
Quarantined rows: 2

Quarantine reasons:
+-------------------------------+-----+
|quarantine_reason              |count|
+-----------------------

In [15]:
#verifying the overall tally of actual row counts of raw data set with clean and quarantinee data
#Also checking whether clean contains any duplicate records
primary_keys = {
    "attendees": "attendee_id",
    "food_orders": "order_id",
    "food_stalls": "stall_id",
    "incidents": "incident_id",
    "performances": "performance_id",
    "stages": "stage_id",
    "ticket_scans": "scan_id",
}

for name in standardized:
    raw_count = raw[name].count()
    clean_count = clean[name].count()
    quarantine_count = quarantine[name].count()

    key = primary_keys[name]

    duplicate_groups = (
        clean[name]
        .groupBy(key)
        .count()
        .filter(F.col(key).isNotNull() & (F.col("count") > 1))
        .count()
    )

    print(
        f"{name:15} "
        f"raw={raw_count:4} "
        f"clean={clean_count:4} "
        f"quarantine={quarantine_count:3} "
        f"reconciles={raw_count == clean_count + quarantine_count} "
        f"duplicate_key_groups={duplicate_groups}"
    )

attendees       raw=1000 clean=1000 quarantine=  0 reconciles=True duplicate_key_groups=0
food_orders     raw=1602 clean=1557 quarantine= 45 reconciles=True duplicate_key_groups=0
food_stalls     raw=   6 clean=   6 quarantine=  0 reconciles=True duplicate_key_groups=0
incidents       raw= 180 clean= 178 quarantine=  2 reconciles=True duplicate_key_groups=0
performances    raw=  30 clean=  28 quarantine=  2 reconciles=True duplicate_key_groups=0
stages          raw=   5 clean=   5 quarantine=  0 reconciles=True duplicate_key_groups=0
ticket_scans    raw=2803 clean=2794 quarantine=  9 reconciles=True duplicate_key_groups=0


In [16]:
def validate_reconciliation(raw_df, clean_df, quarantine_df):
    """Check whether raw rows equal clean rows plus quarantined rows."""
    raw_count = raw_df.count()
    clean_count = clean_df.count()
    quarantine_count = quarantine_df.count()

    return {
        "raw_count": raw_count,
        "clean_count": clean_count,
        "quarantine_count": quarantine_count,
        "passed": raw_count == clean_count + quarantine_count,
    }


# Test the reusable function on every dataset
datasets = [
    "attendees",
    "food_orders",
    "food_stalls",
    "incidents",
    "performances",
    "stages",
    "ticket_scans",
]
for name in datasets:
    result = validate_reconciliation(
        raw[name],
        clean[name],
        quarantine[name]
    )

    print(
        f"{name}: {'PASS' if result['passed'] else 'FAIL'} "
        f"(raw={result['raw_count']}, "
        f"clean={result['clean_count']}, "
        f"quarantine={result['quarantine_count']})"
    )

attendees: PASS (raw=1000, clean=1000, quarantine=0)
food_orders: PASS (raw=1602, clean=1557, quarantine=45)
food_stalls: PASS (raw=6, clean=6, quarantine=0)
incidents: PASS (raw=180, clean=178, quarantine=2)
performances: PASS (raw=30, clean=28, quarantine=2)
stages: PASS (raw=5, clean=5, quarantine=0)
ticket_scans: PASS (raw=2803, clean=2794, quarantine=9)


In [17]:
#displaying records of food_orders that doesnt follow business/validation rules
(
    quarantine["food_orders"]
    .filter(
        F.col("quarantine_reason")
        == "required_field_or_value_invalid"
    )
    .select(
        "order_id",
        "attendee_id",
        "stall_id",
        "order_time",
        "amount",
        "order_status",
        "quarantine_reason"
    )
    .orderBy("order_id")
    .show(50, truncate=False)
)

+--------+-----------+--------+-------------------+------+------------+-------------------------------+
|order_id|attendee_id|stall_id|order_time         |amount|order_status|quarantine_reason              |
+--------+-----------+--------+-------------------+------+------------+-------------------------------+
|O00004  |A00585     |FS03    |2026-09-11 12:28:00|NULL  |cancelled   |required_field_or_value_invalid|
|O00032  |A00455     |FS06    |2026-09-12 22:33:00|-12.50|completed   |required_field_or_value_invalid|
|O00044  |A00582     |FS03    |2026-09-12 23:49:00|NULL  |completed   |required_field_or_value_invalid|
|O00136  |A00708     |FS06    |2026-09-11 13:42:00|NULL  |pending     |required_field_or_value_invalid|
|O00215  |A00573     |FS03    |2026-09-13 16:24:00|NULL  |refunded    |required_field_or_value_invalid|
|O00243  |A00862     |FS06    |2026-09-11 15:38:00|NULL  |pending     |required_field_or_value_invalid|
|O00302  |A00044     |FS05    |2026-09-11 19:10:00|NULL  |cancel

In [18]:
#displaying duplicate records of food_orders
(
    quarantine["food_orders"]
    .filter(F.col("quarantine_reason") == "duplicate_primary_key")
    .select("order_id", "attendee_id", "stall_id", "order_time", "amount", "order_status")
    .orderBy("order_id")
    .show(truncate=False)
)

+--------+-----------+--------+-------------------+------+------------+
|order_id|attendee_id|stall_id|order_time         |amount|order_status|
+--------+-----------+--------+-------------------+------+------------+
|O00006  |A00608     |FS02    |2026-09-13 20:35:00|43.07 |pending     |
|O00006  |A00608     |FS02    |2026-09-13 20:35:00|43.07 |pending     |
|O00056  |A00091     |FS05    |2026-09-11 18:56:00|20.87 |pending     |
|O00056  |A00091     |FS05    |2026-09-11 18:56:00|20.87 |pending     |
+--------+-----------+--------+-------------------+------+------------+



In [19]:
from decimal import Decimal

def test_completed_food_revenue(food_orders):
    completed = food_orders.filter(
        F.col("order_status") == "completed"
    )

    actual = completed.agg(
        F.count("*").alias("order_count"),
        F.sum("amount").alias("revenue")
    ).first()

    assert actual["order_count"] == 591, (
        f"Expected 591 completed orders, got {actual['order_count']}"
    )

    assert actual["revenue"] == Decimal("17728.68"), (
        f"Expected revenue 17728.68, got {actual['revenue']}"
    )

    return "PASS: completed food revenue is correct"


print(test_completed_food_revenue(clean["food_orders"]))

PASS: completed food revenue is correct


# **Repeated ticket-scan analysis**

In [20]:
ticket_scans = clean['ticket_scans']
print("clean ticket scans: ",ticket_scans.count())
ticket_scans.groupBy("scan_status").count().orderBy(F.desc("count")).show()

clean ticket scans:  2794
+-----------+-----+
|scan_status|count|
+-----------+-----+
|      valid| 2376|
|     denied|  175|
|  duplicate|  166|
|  cancelled|   77|
+-----------+-----+



In [21]:
scans_per_attendee = (ticket_scans.groupBy("attendee_id")
                      .agg(
                          F.count("*").alias("total_scans"),
                          F.countDistinct("stage_id").alias("distinct_stages"),
                          #gives distinct stage count they scanned at
                          F.countDistinct("scan_time").alias("distinct_scans_times")
                          #gives distinct times they have scanne at
    
                      )
                      .orderBy(F.desc("total_scans")))
scans_per_attendee.show(20,truncate=False)

+-----------+-----------+---------------+--------------------+
|attendee_id|total_scans|distinct_stages|distinct_scans_times|
+-----------+-----------+---------------+--------------------+
|A00346     |10         |4              |10                  |
|A00321     |9          |5              |9                   |
|A00966     |9          |4              |9                   |
|A00665     |8          |5              |8                   |
|A00149     |8          |4              |8                   |
|A00037     |8          |4              |8                   |
|A00270     |8          |4              |8                   |
|A00104     |8          |4              |8                   |
|A00158     |8          |5              |8                   |
|A00257     |8          |5              |8                   |
|A00963     |7          |3              |7                   |
|A00024     |7          |3              |7                   |
|A00756     |7          |4              |7             

In [22]:
same_time_repeats = (
    ticket_scans.groupBy("attendee_id","stage_id","scan_time")
    .agg(F.count("*").alias("repeat_scans"))
    .filter(F.col("repeat_scans")>1)
    .orderBy(F.desc("repeat_scans"))
)
same_time_repeats.show(30,truncate=False)

+-----------+--------+---------+------------+
|attendee_id|stage_id|scan_time|repeat_scans|
+-----------+--------+---------+------------+
+-----------+--------+---------+------------+



In [23]:

# Using  successful scans for this analysis
valid_scans = ticket_scans.filter(
    F.col("scan_status") == "valid"
)

# Compare each scan with the previous scan
# for the same attendee at the same stage
scan_window = (
    Window.partitionBy("attendee_id","stage_id").orderBy("scan_time")
)

consecutive_scans = (
    valid_scans
    .withColumn(
        "previous_scan_time",
        F.lag("scan_time").over(scan_window)
    )
    .withColumn(
        "minutes_since_previous",
        (
            F.col("scan_time").cast("long")
            - F.col("previous_scan_time").cast("long")
        ) / 60
    )
)

# Flag scans within 10 minutes of the previous scan
potential_repeats = (
    consecutive_scans
    .filter(
        F.col("minutes_since_previous").between(0, 10)
    )
    .orderBy("attendee_id", "stage_id", "scan_time")
)

potential_repeats.select(
    "attendee_id",
    "stage_id",
    "previous_scan_time",
    "scan_time",
    "minutes_since_previous"
).show(50, truncate=False)


+-----------+--------+-------------------+-------------------+----------------------+
|attendee_id|stage_id|previous_scan_time |scan_time          |minutes_since_previous|
+-----------+--------+-------------------+-------------------+----------------------+
|A00006     |ST03    |2026-09-13 15:07:00|2026-09-13 15:12:00|5.0                   |
|A00151     |ST01    |2026-09-11 16:51:00|2026-09-11 16:59:00|8.0                   |
|A00432     |ST04    |2026-09-12 17:08:00|2026-09-12 17:12:00|4.0                   |
|A00631     |ST04    |2026-09-11 13:57:00|2026-09-11 14:00:00|3.0                   |
|A00639     |ST04    |2026-09-13 13:00:00|2026-09-13 13:07:00|7.0                   |
+-----------+--------+-------------------+-------------------+----------------------+



In [24]:

potential_repeats.groupBy("stage_id").agg(
    F.count("*").alias("close_repeat_events"),
    F.round(
        F.avg("minutes_since_previous"), 2
    ).alias("avg_minutes_between_scans"),
    F.min("minutes_since_previous").alias("min_gap_minutes")
).orderBy(
    F.desc("close_repeat_events")
).show(truncate=False)


+--------+-------------------+-------------------------+---------------+
|stage_id|close_repeat_events|avg_minutes_between_scans|min_gap_minutes|
+--------+-------------------+-------------------------+---------------+
|ST04    |3                  |4.67                     |3.0            |
|ST01    |1                  |8.0                      |8.0            |
|ST03    |1                  |5.0                      |5.0            |
+--------+-------------------+-------------------------+---------------+



# **Stage activity and capacity**

In [25]:
stages= clean['stages']
stages.select("stage_id",'stage_name','capacity').orderBy(F.desc('capacity')).show(truncate=False)

+--------+-------------+--------+
|stage_id|stage_name   |capacity|
+--------+-------------+--------+
|ST01    |Main Stage   |8500    |
|ST02    |River Stage  |4200    |
|ST03    |Sunset Tent  |2600    |
|ST04    |Indie Corner |1800    |
|ST05    |Rooftop Beats|1200    |
+--------+-------------+--------+



In [26]:
valid_ticket_scans = ticket_scans.filter(F.col("scan_status")=="valid")

valid_scans_by_stage = (
    valid_ticket_scans.groupBy("stage_id")
    .agg(
        F.count("*").alias("valid_scan_count"),
        F.countDistinct("attendee_id").alias("unique_attendees")
    )
)

valid_scans_by_stage.orderBy(F.desc('valid_scan_count')).show(truncate=False)


+--------+----------------+----------------+
|stage_id|valid_scan_count|unique_attendees|
+--------+----------------+----------------+
|ST04    |519             |419             |
|ST01    |494             |395             |
|ST02    |471             |379             |
|ST03    |461             |382             |
|ST05    |431             |353             |
+--------+----------------+----------------+



In [27]:
# Inspect the physical execution plan for stage activity
print("=== Stage Activity: Physical Execution Plan ===")

valid_scans_by_stage.explain(mode="formatted")

=== Stage Activity: Physical Execution Plan ===
== Physical Plan ==
AdaptiveSparkPlan (37)
+- HashAggregate (36)
   +- Exchange (35)
      +- HashAggregate (34)
         +- HashAggregate (33)
            +- Exchange (32)
               +- HashAggregate (31)
                  +- BroadcastHashJoin LeftSemi BuildRight (30)
                     :- BroadcastHashJoin LeftSemi BuildRight (22)
                     :  :- Project (14)
                     :  :  +- BroadcastHashJoin LeftAnti BuildRight (13)
                     :  :     :- Project (3)
                     :  :     :  +- Filter (2)
                     :  :     :     +- Scan csv  (1)
                     :  :     +- BroadcastExchange (12)
                     :  :        +- Project (11)
                     :  :           +- Filter (10)
                     :  :              +- HashAggregate (9)
                     :  :                 +- Exchange (8)
                     :  :                    +- HashAggregate (7)
             

In [28]:

stage_activity = (
    stages
    .join(valid_scans_by_stage, on="stage_id", how="left")
    .fillna({
        "valid_scan_count": 0,
        "unique_attendees": 0
    })
    .withColumn(
        "scans_per_capacity",
        F.round(
            F.col("valid_scan_count") / F.col("capacity") * 100,
            2
        )
    )
    .orderBy(F.desc("scans_per_capacity"))
)

stage_activity.select(
    "stage_id",
    "stage_name",
    "capacity",
    "valid_scan_count",
    "unique_attendees",
    "scans_per_capacity"
).show(truncate=False)


+--------+-------------+--------+----------------+----------------+------------------+
|stage_id|stage_name   |capacity|valid_scan_count|unique_attendees|scans_per_capacity|
+--------+-------------+--------+----------------+----------------+------------------+
|ST05    |Rooftop Beats|1200    |431             |353             |35.92             |
|ST04    |Indie Corner |1800    |519             |419             |28.83             |
|ST03    |Sunset Tent  |2600    |461             |382             |17.73             |
|ST02    |River Stage  |4200    |471             |379             |11.21             |
|ST01    |Main Stage   |8500    |494             |395             |5.81              |
+--------+-------------+--------+----------------+----------------+------------------+



In [29]:

daily_stage_activity = (
    valid_ticket_scans
    .withColumn("festival_date", F.to_date("scan_time"))
    .groupBy("festival_date", "stage_id")
    .agg(
        F.count("*").alias("valid_scan_count"),
        F.countDistinct("attendee_id").alias("unique_attendees")
    )
    .join(
        stages.select("stage_id", "stage_name", "capacity"),
        on="stage_id",
        how="left"
    )
    .orderBy("festival_date", F.desc("valid_scan_count"))
)

daily_stage_activity.show(50, truncate=False)


+--------+-------------+----------------+----------------+-------------+--------+
|stage_id|festival_date|valid_scan_count|unique_attendees|stage_name   |capacity|
+--------+-------------+----------------+----------------+-------------+--------+
|ST04    |2026-09-11   |169             |158             |Indie Corner |1800    |
|ST01    |2026-09-11   |167             |154             |Main Stage   |8500    |
|ST03    |2026-09-11   |153             |141             |Sunset Tent  |2600    |
|ST02    |2026-09-11   |153             |139             |River Stage  |4200    |
|ST05    |2026-09-11   |132             |126             |Rooftop Beats|1200    |
|ST02    |2026-09-12   |162             |153             |River Stage  |4200    |
|ST04    |2026-09-12   |159             |149             |Indie Corner |1800    |
|ST01    |2026-09-12   |158             |150             |Main Stage   |8500    |
|ST05    |2026-09-12   |149             |138             |Rooftop Beats|1200    |
|ST03    |2026-0

In [30]:
# Register the cleaned ticket scans for Spark SQL
clean["ticket_scans"].createOrReplaceTempView("ticket_scans_clean")

# 1. Spark SQL analysis
stage_activity_sql = spark.sql("""
    SELECT
        stage_id,
        COUNT(*) AS valid_scan_count,
        COUNT(DISTINCT attendee_id) AS unique_attendees
    FROM ticket_scans_clean
    WHERE scan_status = 'valid'
    GROUP BY stage_id
""")

print("Stage activity using Spark SQL:")
stage_activity_sql.orderBy("stage_id").show(truncate=False)


# 2. Compare against the existing DataFrame API result
sql_result = stage_activity_sql.select(
    "stage_id", "valid_scan_count", "unique_attendees"
)

dataframe_result = valid_scans_by_stage.select(
    "stage_id", "valid_scan_count", "unique_attendees"
)

sql_only = sql_result.exceptAll(dataframe_result)
dataframe_only = dataframe_result.exceptAll(sql_result)

print("Rows present only in SQL:", sql_only.count())
print("Rows present only in DataFrame API:", dataframe_only.count())

if sql_only.count() == 0 and dataframe_only.count() == 0:
    print("PASS: Spark SQL and DataFrame API results match.")
else:
    print("CHECK: The two results differ.")

Stage activity using Spark SQL:
+--------+----------------+----------------+
|stage_id|valid_scan_count|unique_attendees|
+--------+----------------+----------------+
|ST01    |494             |395             |
|ST02    |471             |379             |
|ST03    |461             |382             |
|ST04    |519             |419             |
|ST05    |431             |353             |
+--------+----------------+----------------+

Rows present only in SQL: 0
Rows present only in DataFrame API: 0
PASS: Spark SQL and DataFrame API results match.


In [31]:
def test_stage_activity(valid_scans, valid_scans_by_stage):
    # Check total valid scan events
    actual_total = valid_scans.count()
    assert actual_total == 2376, (
        f"Expected 2376 valid scans, got {actual_total}"
    )

    # Check that stage-level aggregation matches source data
    expected = (
        valid_scans
        .groupBy("stage_id")
        .agg(
            F.count("*").alias("valid_scan_count"),
            F.countDistinct("attendee_id").alias("unique_attendees")
        )
    )

    differences = (
        expected.exceptAll(valid_scans_by_stage)
        .unionByName(valid_scans_by_stage.exceptAll(expected))
        .count()
    )

    assert differences == 0, (
        f"Stage-level results differ in {differences} rows"
    )

    return "PASS: valid scan total and stage-level activity are correct"


print(test_stage_activity(valid_scans, valid_scans_by_stage))

PASS: valid scan total and stage-level activity are correct


In [32]:
performances = clean['performances']

performance_activity = (
    performances
    .withColumn(
        "festival_date",
        F.to_date("start_time")
    )
    .join(
        daily_stage_activity,
        on=["stage_id", "festival_date"],
        how="left"
    )
    .select(
        "performance_id",
        "artist_name",
        "stage_id",
        "stage_name",
        "start_time",
        "end_time",
        "festival_date",
        "valid_scan_count",
        "unique_attendees",
        "capacity"
    )
    .orderBy("festival_date", "stage_id", "start_time")
)

performance_activity.show(30, truncate=False)

+--------------+--------------------+--------+-------------+-------------------+-------------------+-------------+----------------+----------------+--------+
|performance_id|artist_name         |stage_id|stage_name   |start_time         |end_time           |festival_date|valid_scan_count|unique_attendees|capacity|
+--------------+--------------------+--------+-------------+-------------------+-------------------+-------------+----------------+----------------+--------+
|P001          |Static Bloom        |ST01    |Main Stage   |2026-09-11 12:30:00|2026-09-11 13:30:00|2026-09-11   |167             |154             |8500    |
|P003          |Static Bloom        |ST01    |Main Stage   |2026-09-11 13:15:00|2026-09-11 14:00:00|2026-09-11   |167             |154             |8500    |
|P010          |The Paper Satellites|ST01    |Main Stage   |2026-09-11 15:30:00|2026-09-11 16:30:00|2026-09-11   |167             |154             |8500    |
|P007          |Nova June           |ST01    |Main S

In [33]:


p1 = performances.alias("p1")
p2 = performances.alias("p2")

overlapping_performances = (
    p1.join(
        p2,
        (F.col("p1.stage_id") == F.col("p2.stage_id")) &
        (F.col("p1.performance_id") < F.col("p2.performance_id")) &
        (F.col("p1.start_time") < F.col("p2.end_time")) &
        (F.col("p2.start_time") < F.col("p1.end_time")),
        "inner"
    )
    .select(
        F.col("p1.stage_id").alias("stage_id"),
        F.col("p1.performance_id").alias("performance_1"),
        F.col("p1.artist_name").alias("artist_1"),
        F.col("p1.start_time").alias("start_1"),
        F.col("p1.end_time").alias("end_1"),
        F.col("p2.performance_id").alias("performance_2"),
        F.col("p2.artist_name").alias("artist_2"),
        F.col("p2.start_time").alias("start_2"),
        F.col("p2.end_time").alias("end_2")
    )
    .orderBy("stage_id", "start_1")
)

overlapping_performances.show(100, truncate=False)

+--------+-------------+--------------------+-------------------+-------------------+-------------+----------------+-------------------+-------------------+
|stage_id|performance_1|artist_1            |start_1            |end_1              |performance_2|artist_2        |start_2            |end_2              |
+--------+-------------+--------------------+-------------------+-------------------+-------------+----------------+-------------------+-------------------+
|ST01    |P001         |Static Bloom        |2026-09-11 12:30:00|2026-09-11 13:30:00|P003         |Static Bloom    |2026-09-11 13:15:00|2026-09-11 14:00:00|
|ST03    |P021         |The Paper Satellites|2026-09-13 20:00:00|2026-09-13 21:15:00|P023         |The Wild Signals|2026-09-13 20:15:00|2026-09-13 21:00:00|
|ST04    |P011         |Velvet Echo         |2026-09-12 16:45:00|2026-09-12 17:45:00|P018         |Mira Sol        |2026-09-12 17:15:00|2026-09-12 18:45:00|
|ST04    |P011         |Velvet Echo         |2026-09-12 16

# **Food-stall performance**

In [34]:
food_orders = clean['food_orders']
food_stalls = clean['food_stalls']

food_orders.groupBy('order_status').count().orderBy(F.desc("count")).show(truncate=False)

+------------+-----+
|order_status|count|
+------------+-----+
|completed   |591  |
|pending     |361  |
|cancelled   |308  |
|refunded    |297  |
+------------+-----+



In [35]:
# Check amount quality by order status
food_orders.groupBy("order_status").agg(
    F.count("*").alias("order_count"),
    F.sum(
        F.when(F.col("amount").isNull(), 1).otherwise(0)
    ).alias("missing_amounts"),
    F.sum(
        F.when(F.col("amount") <= 0, 1).otherwise(0)
    ).alias("non_positive_amounts"),
    F.sum("amount").alias("amount_total")
).orderBy(F.desc('amount_total')).show(truncate=False)

+------------+-----------+---------------+--------------------+------------+
|order_status|order_count|missing_amounts|non_positive_amounts|amount_total|
+------------+-----------+---------------+--------------------+------------+
|completed   |591        |0              |0                   |17728.68    |
|pending     |361        |0              |0                   |11305.86    |
|cancelled   |308        |0              |0                   |9006.27     |
|refunded    |297        |0              |0                   |8611.22     |
+------------+-----------+---------------+--------------------+------------+



In [36]:
completed_orders = food_orders.filter(F.col("order_status")=="completed")
completed_orders.agg(
    F.count("*").alias("completed_order_count"),
    F.sum("amount").alias("total_completed_revenue"),
    F.avg("amount").alias("average_order_value"),
    F.min("amount").alias("minimum_order_amount"),
    F.max("amount").alias("maximum_order_amount")
).show(truncate=False)



+---------------------+-----------------------+-------------------+--------------------+--------------------+
|completed_order_count|total_completed_revenue|average_order_value|minimum_order_amount|maximum_order_amount|
+---------------------+-----------------------+-------------------+--------------------+--------------------+
|591                  |17728.68               |29.997766          |4.04                |54.97               |
+---------------------+-----------------------+-------------------+--------------------+--------------------+



In [37]:
stall_revenue = ((
    completed_orders.groupBy("stall_id")
    .agg(
        F.count("*").alias("completed_orders"),
        F.sum("amount").alias("total_revenue"),
        F.avg("amount").alias("average_order_value"),
        F.count(
            F.when(F.col("amount").isNull(),1)
        ).alias("missing_amounts")
    ))
    .join(food_stalls,
          on='stall_id',
          how='left'
          )
    .select("stall_id","stall_name","cuisine","completed_orders",
            "total_revenue","average_order_value","missing_amounts")
    .orderBy(F.desc('total_revenue'))
)

stall_revenue.show(truncate=False)
    


+--------+------------+---------+----------------+-------------+-------------------+---------------+
|stall_id|stall_name  |cuisine  |completed_orders|total_revenue|average_order_value|missing_amounts|
+--------+------------+---------+----------------+-------------+-------------------+---------------+
|FS02    |Noodle Lab  |asian    |113             |3312.82      |29.316991          |0              |
|FS05    |Sweet Static|dessert  |97              |3084.17      |31.795567          |0              |
|FS01    |Spice Route |indian   |103             |3067.49      |29.781456          |0              |
|FS04    |Burger Orbit|fast food|105             |2965.42      |28.242095          |0              |
|FS06    |Taco Tempo  |mexican  |85              |2741.56      |32.253647          |0              |
|FS03    |Green Fork  |vegan    |88              |2557.22      |29.059318          |0              |
+--------+------------+---------+----------------+-------------+-------------------+-------

In [38]:
daily_food_revenue = (
    completed_orders
    .withColumn("order_date", F.to_date("order_time"))
    .groupBy("order_date")
    .agg(
        F.count("*").alias("completed_orders"),
        F.sum("amount").alias("daily_revenue")
    )
    .orderBy("order_date")
)

daily_food_revenue.show(truncate=False)

+----------+----------------+-------------+
|order_date|completed_orders|daily_revenue|
+----------+----------------+-------------+
|2026-09-11|209             |6335.62      |
|2026-09-12|201             |5918.72      |
|2026-09-13|181             |5474.34      |
+----------+----------------+-------------+



In [39]:
# Register the cleaned food orders DataFrame as a temporary SQL view
food_orders.createOrReplaceTempView("food_orders_clean")

# 1. Perform the analysis using Spark SQL
daily_revenue_sql = spark.sql("""
    SELECT
        TO_DATE(order_time) AS order_date,
        COUNT(*) AS completed_orders,
        SUM(amount) AS daily_revenue
    FROM food_orders_clean
    WHERE order_status = 'completed'
    GROUP BY TO_DATE(order_time)
    ORDER BY order_date
""")

print("Results using Spark SQL:")
daily_revenue_sql.show(truncate=False)


# 2. Compare SQL results against the existing DataFrame API results
# daily_food_revenue was created in your existing cell 35.

sql_result = daily_revenue_sql.select(
    "order_date", "completed_orders", "daily_revenue"
)

dataframe_result = daily_food_revenue.select(
    "order_date", "completed_orders", "daily_revenue"
)

# Check for rows present in SQL but missing from the DataFrame result,
# and vice versa.
sql_only = sql_result.exceptAll(dataframe_result)
dataframe_only = dataframe_result.exceptAll(sql_result)

print("Rows present only in SQL:", sql_only.count())
print("Rows present only in DataFrame API:", dataframe_only.count())

if sql_only.count() == 0 and dataframe_only.count() == 0:
    print("PASS: Spark SQL and DataFrame API results match.")
else:
    print("CHECK: The two results differ.")

Results using Spark SQL:
+----------+----------------+-------------+
|order_date|completed_orders|daily_revenue|
+----------+----------------+-------------+
|2026-09-11|209             |6335.62      |
|2026-09-12|201             |5918.72      |
|2026-09-13|181             |5474.34      |
+----------+----------------+-------------+

Rows present only in SQL: 0
Rows present only in DataFrame API: 0
PASS: Spark SQL and DataFrame API results match.


# **Peak periods and incidents**

In [40]:
incidents = clean['incidents']
incidents.printSchema()


incidents.groupBy("stage_id") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(truncate=False)

print()

incidents.groupBy("incident_type","severity")\
         .count()\
         .orderBy(F.desc("count"))\
         .show(30,truncate=False)



root
 |-- incident_id: string (nullable = true)
 |-- stage_id: string (nullable = true)
 |-- incident_time: timestamp (nullable = true)
 |-- incident_type: string (nullable = true)
 |-- severity: string (nullable = true)

+--------+-----+
|stage_id|count|
+--------+-----+
|ST02    |50   |
|ST03    |37   |
|ST05    |36   |
|ST04    |30   |
|ST01    |25   |
+--------+-----+


+-------------+--------+-----+
|incident_type|severity|count|
+-------------+--------+-----+
|security     |high    |11   |
|equipment    |medium  |10   |
|noise        |medium  |10   |
|medical      |low     |9    |
|medical      |medium  |9    |
|queue        |medium  |9    |
|security     |medium  |8    |
|equipment    |low     |7    |
|lost_person  |low     |7    |
|security     |critical|7    |
|medical      |NULL    |7    |
|equipment    |critical|6    |
|lost_person  |NULL    |6    |
|lost_person  |medium  |6    |
|equipment    |high    |5    |
|security     |NULL    |5    |
|queue        |critical|5    |
|no

In [41]:
incidents.groupBy("severity").count().orderBy(
    F.desc("count")
).show(truncate=False)

incidents.filter(
    F.col("severity").isNull()
).groupBy("incident_type").count().orderBy(
    F.desc("count")
).show(truncate=False)

+--------+-----+
|severity|count|
+--------+-----+
|medium  |52   |
|low     |33   |
|critical|32   |
|NULL    |31   |
|high    |30   |
+--------+-----+

+-------------+-----+
|incident_type|count|
+-------------+-----+
|medical      |7    |
|lost_person  |6    |
|security     |5    |
|noise        |5    |
|queue        |4    |
|equipment    |4    |
+-------------+-----+



In [42]:
incident_severity_quality = (
    incidents
    .groupBy("incident_type")
    .agg(
        F.count("*").alias("total_incidents"),
        F.sum(
            F.when(F.col("severity").isNull(), 1).otherwise(0)
        ).alias("missing_severity_count")
    )
    .withColumn(
        "missing_severity_pct",
        F.round(
            F.col("missing_severity_count") /
            F.col("total_incidents") * 100,
            2
        )
    )
    .orderBy(F.desc("missing_severity_pct"))
)

incident_severity_quality.show(truncate=False)

+-------------+---------------+----------------------+--------------------+
|incident_type|total_incidents|missing_severity_count|missing_severity_pct|
+-------------+---------------+----------------------+--------------------+
|medical      |31             |7                     |22.58               |
|lost_person  |27             |6                     |22.22               |
|queue        |23             |4                     |17.39               |
|noise        |29             |5                     |17.24               |
|security     |36             |5                     |13.89               |
|equipment    |32             |4                     |12.5                |
+-------------+---------------+----------------------+--------------------+



In [43]:
incident_by_stage = (
    incidents.groupBy("stage_id")
    .agg(F.count("*").alias("incident_count"))
)

incident_activity = (
    stages
    .join(incident_by_stage, on="stage_id", how="left")
    .join(
        valid_scans_by_stage.select("stage_id", "valid_scan_count"),
        on="stage_id",
        how="left"
    )
    .fillna({
        "incident_count": 0,
        "valid_scan_count": 0
    })
    .withColumn(
        "incidents_per_1000_valid_scans",
        F.round(
            F.col("incident_count") /
            F.when(
                F.col("valid_scan_count") > 0,
                F.col("valid_scan_count")
            ) * 1000,
            2
        )
    )
    .select(
        "stage_id",
        "stage_name",
        "capacity",
        "incident_count",
        "valid_scan_count",
        "incidents_per_1000_valid_scans"
    )
    .orderBy(F.desc("incidents_per_1000_valid_scans"))
)

incident_activity.show(truncate=False)

+--------+-------------+--------+--------------+----------------+------------------------------+
|stage_id|stage_name   |capacity|incident_count|valid_scan_count|incidents_per_1000_valid_scans|
+--------+-------------+--------+--------------+----------------+------------------------------+
|ST02    |River Stage  |4200    |50            |471             |106.16                        |
|ST05    |Rooftop Beats|1200    |36            |431             |83.53                         |
|ST03    |Sunset Tent  |2600    |37            |461             |80.26                         |
|ST04    |Indie Corner |1800    |30            |519             |57.8                          |
|ST01    |Main Stage   |8500    |25            |494             |50.61                         |
+--------+-------------+--------+--------------+----------------+------------------------------+



In [44]:
# Register the cleaned incidents DataFrame
incidents.createOrReplaceTempView("incidents_clean")

# 1. Spark SQL analysis
incident_counts_sql = spark.sql("""
    SELECT
        stage_id,
        COUNT(*) AS incident_count
    FROM incidents_clean
    GROUP BY stage_id
""")

print("Incident counts using Spark SQL:")
incident_counts_sql.orderBy("stage_id").show(truncate=False)


# 2. Equivalent DataFrame API analysis
incident_counts_df = (
    incidents
    .groupBy("stage_id")
    .count()
    .withColumnRenamed("count", "incident_count")
)

# 3. Compare both results
sql_result = incident_counts_sql.select("stage_id", "incident_count")
dataframe_result = incident_counts_df.select("stage_id", "incident_count")

sql_only = sql_result.exceptAll(dataframe_result)
dataframe_only = dataframe_result.exceptAll(sql_result)

sql_diff_count = sql_only.count()
df_diff_count = dataframe_only.count()

print("Rows present only in SQL:", sql_diff_count)
print("Rows present only in DataFrame API:", df_diff_count)

if sql_diff_count == 0 and df_diff_count == 0:
    print("PASS: Spark SQL and DataFrame API results match.")
else:
    print("CHECK: The two results differ.")

Incident counts using Spark SQL:
+--------+--------------+
|stage_id|incident_count|
+--------+--------------+
|ST01    |25            |
|ST02    |50            |
|ST03    |37            |
|ST04    |30            |
|ST05    |36            |
+--------+--------------+

Rows present only in SQL: 0
Rows present only in DataFrame API: 0
PASS: Spark SQL and DataFrame API results match.


In [45]:
incident_summary = (
    incidents
    .agg(
        F.count("*").alias("total_incidents"),
        F.sum(
            F.when(F.col("severity").isNull(), 1).otherwise(0)
        ).alias("missing_severity"),
        F.sum(
            F.when(F.col("severity") == "critical", 1).otherwise(0)
        ).alias("critical_incidents"),
        F.sum(
            F.when(F.col("severity") == "high", 1).otherwise(0)
        ).alias("high_incidents")
    )
    .withColumn(
        "missing_severity_pct",
        F.round(
            F.col("missing_severity") /
            F.col("total_incidents") * 100,
            2
        )
    )
)

incident_summary.show(truncate=False)

+---------------+----------------+------------------+--------------+--------------------+
|total_incidents|missing_severity|critical_incidents|high_incidents|missing_severity_pct|
+---------------+----------------+------------------+--------------+--------------------+
|178            |31              |32                |30            |17.42               |
+---------------+----------------+------------------+--------------+--------------------+



# **Performances Analysis**

In [46]:
performance_scan_matches = (
    performances.alias("p")
    .join(
        valid_scans.alias("s"),
        (F.col("p.stage_id") == F.col("s.stage_id")) &
        (F.col("s.scan_time") >= F.col("p.start_time")) &
        (F.col("s.scan_time") < F.col("p.end_time")),
        "left"
    )
    .select(
        F.col("p.performance_id").alias("performance_id"),
        F.col("p.artist_name").alias("artist_name"),
        F.col("p.stage_id").alias("stage_id"),
        F.col("p.start_time").alias("start_time"),
        F.col("p.end_time").alias("end_time"),
        F.col("p.genre").alias("genre"),
        F.col("s.scan_id").alias("matched_scan_id"),
        F.col("s.attendee_id").alias("matched_attendee_id")
    )
    .groupBy(
        "performance_id",
        "artist_name",
        "stage_id",
        "start_time",
        "end_time",
        "genre"
    )
    .agg(
        F.count("matched_scan_id").alias("valid_scan_events"),
        F.countDistinct("matched_attendee_id").alias(
            "attendance_proxy"
        )
    )
)

performance_scan_matches.orderBy(
    F.desc("attendance_proxy")
).show(30, truncate=False)

+--------------+--------------------+--------+-------------------+-------------------+----------+-----------------+----------------+
|performance_id|artist_name         |stage_id|start_time         |end_time           |genre     |valid_scan_events|attendance_proxy|
+--------------+--------------------+--------+-------------------+-------------------+----------+-----------------+----------------+
|P030          |The Wild Signals    |ST04    |2026-09-13 17:45:00|2026-09-13 19:15:00|indie     |32               |32              |
|P002          |Nova June           |ST01    |2026-09-11 20:00:00|2026-09-11 21:30:00|electronic|30               |30              |
|P027          |The Paper Satellites|ST02    |2026-09-13 21:00:00|2026-09-13 22:30:00|indie     |27               |27              |
|P021          |The Paper Satellites|ST03    |2026-09-13 20:00:00|2026-09-13 21:15:00|electronic|22               |22              |
|P005          |Midnight Arcade     |ST03    |2026-09-11 12:15:00|202

In [47]:
print("=== Performance Attendance Proxy: Physical Execution Plan ===")

performance_scan_matches.explain(mode="formatted")

=== Performance Attendance Proxy: Physical Execution Plan ===
== Physical Plan ==
AdaptiveSparkPlan (60)
+- HashAggregate (59)
   +- Exchange (58)
      +- HashAggregate (57)
         +- HashAggregate (56)
            +- Exchange (55)
               +- HashAggregate (54)
                  +- Project (53)
                     +- BroadcastHashJoin LeftOuter BuildRight (52)
                        :- BroadcastHashJoin LeftSemi BuildRight (21)
                        :  :- BroadcastHashJoin LeftAnti BuildRight (13)
                        :  :  :- Project (3)
                        :  :  :  +- Filter (2)
                        :  :  :     +- Scan csv  (1)
                        :  :  +- BroadcastExchange (12)
                        :  :     +- Project (11)
                        :  :        +- Filter (10)
                        :  :           +- HashAggregate (9)
                        :  :              +- Exchange (8)
                        :  :                 +- HashAggregate (7

In [48]:

attendance_window = Window.orderBy(
    F.desc("attendance_proxy")
)

performance_rankings = (
    performance_scan_matches
    .withColumn(
        "row_number",
        F.row_number().over(
            Window.orderBy(
                F.desc("attendance_proxy"),
                F.asc("performance_id")
            )
        )
    )
    .withColumn(
        "rank",
        F.rank().over(attendance_window)
    )
    .withColumn(
        "dense_rank",
        F.dense_rank().over(attendance_window)
    )
    .select(
        "row_number",
        "rank",
        "dense_rank",
        "performance_id",
        "artist_name",
        "genre",
        "stage_id",
        "attendance_proxy"
    )
    .orderBy("row_number")
)

performance_rankings.show(50, truncate=False)

+----------+----+----------+--------------+--------------------+----------+--------+----------------+
|row_number|rank|dense_rank|performance_id|artist_name         |genre     |stage_id|attendance_proxy|
+----------+----+----------+--------------+--------------------+----------+--------+----------------+
|1         |1   |1         |P030          |The Wild Signals    |indie     |ST04    |32              |
|2         |2   |2         |P002          |Nova June           |electronic|ST01    |30              |
|3         |3   |3         |P027          |The Paper Satellites|indie     |ST02    |27              |
|4         |4   |4         |P021          |The Paper Satellites|electronic|ST03    |22              |
|5         |5   |5         |P005          |Midnight Arcade     |hip-hop   |ST03    |20              |
|6         |5   |5         |P013          |Midnight Arcade     |pop       |ST02    |20              |
|7         |7   |6         |P016          |Mira Sol            |pop       |ST04   

In [49]:
genre_analysis = (
    performance_scan_matches
    .groupBy("genre")
    .agg(
        F.count("*").alias("performance_count"),
        F.round(
            F.avg("attendance_proxy"), 2
        ).alias("avg_attendance_proxy"),
        F.max("attendance_proxy").alias("highest_proxy"),
        F.min("attendance_proxy").alias("lowest_proxy")
    )
    .orderBy(F.desc("avg_attendance_proxy"))
)

genre_analysis.show(truncate=False)

+----------+-----------------+--------------------+-------------+------------+
|genre     |performance_count|avg_attendance_proxy|highest_proxy|lowest_proxy|
+----------+-----------------+--------------------+-------------+------------+
|indie     |5                |19.4                |32           |8           |
|electronic|7                |17.0                |30           |7           |
|hip-hop   |6                |15.5                |20           |12          |
|rock      |2                |15.5                |19           |12          |
|pop       |7                |14.71               |20           |7           |
|NULL      |1                |8.0                 |8            |8           |
+----------+-----------------+--------------------+-------------+------------+



In [50]:
performances.agg(
    F.count("*").alias("performance_rows"),
    F.countDistinct("performance_id").alias("distinct_performance_ids"),
    F.sum(
        F.when(F.col("genre").isNull(), 1).otherwise(0)
    ).alias("missing_genres")
).show()

+----------------+------------------------+--------------+
|performance_rows|distinct_performance_ids|missing_genres|
+----------------+------------------------+--------------+
|              28|                      28|             1|
+----------------+------------------------+--------------+

